In [1]:
from oqd_core.interface.analog import MathAdd, MathNum

a = MathNum(value=1) + MathNum(value=2)

print(a)

s = a.model_dump_json()

b = MathAdd.model_validate_json(s)

print(b)

class_='MathAdd' expr1=MathNum(class_='MathNum', value=1) expr2=MathNum(class_='MathNum', value=2)
class_='MathAdd' expr1=MathNum(class_='MathNum', value=1) expr2=MathNum(class_='MathNum', value=2)


In [2]:
from oqd_compiler_infrastructure import Post, PrettyPrint

from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog

printer = Post(PrettyPrint())

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)


In [3]:
# from oqd_core.frontend.analog import parse_analog

# program = "r = qreg(2) \n H_single = 0.5 %* %X %+ 0.5 %* %Z \n evolve(H_single, 1.0, r[0])"
# circuit = parse_analog(program)

In [4]:
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table
import json

symbol_table = str(symbol_table)
print(json.dumps(symbol_table, indent=2))

"in_env={0: {}, 1: {}, 2: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=(2, 0), list_elem=None)}, 3: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=(2, 0), list_elem=None), 's': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TMReg'>, target_dim=(0, 3), list_elem=None)}, 4: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=(2, 0), list_elem=None), 's': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TMReg'>, target_dim=(0, 3), list_elem=None), 'q0': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQRef'>, target_dim=(1, 0), list_elem=None)}, 5: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=(2, 0), list_elem=None), 's': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TMReg'>, target_dim=(0, 3), list_elem=None), 'q0': SymbolBinding(lattice_t

In [5]:
from oqd_core.analysis.analog import (
    AnalogCFGBuilder,
    AnalogSymbolTableBuilder,
    AnalogTypeChecker,
)
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
type_checker = AnalogTypeChecker(cfg)
dataflow_result = type_checker.dataflow_result
symbol_analysis = AnalogSymbolTableBuilder(cfg, dataflow_result)
symbol_table = symbol_analysis.symbol_table
circuit, cfg = compile_analog_circuit(circuit, cfg, symbol_table)

In [6]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'r',
   'value': {'class_': 'QuantumRegister', 'size': 2}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 's',
   'value': {'class_': 'ModeRegister', 'size': 3}},
  'preds': [1],
  'succs': [3],
  'exit_nodes': [],
  'edge_labels': {}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q0',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 0}},
  'preds': [2],
  'succs': [4],
  'exit_nodes': [],
  'edge_labels': {}},
 4: {'register_id': 4,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q1',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access'

In [7]:
circuit

AnalogCircuit(class_='AnalogCircuit', statements=[Declaration(class_='Declaration', name='r', value=QuantumRegister(class_='QuantumRegister', size=2)), Declaration(class_='Declaration', name='s', value=ModeRegister(class_='ModeRegister', size=3)), Declaration(class_='Declaration', name='q0', value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=0)), Declaration(class_='Declaration', name='q1', value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=1)), Declaration(class_='Declaration', name='targets', value=AnalogList(class_='AnalogList', values=[Access(class_='Access', name='q0'), Access(class_='Access', name='q1')])), Declaration(class_='Declaration', name='pi', value=MathNum(class_='MathNum', value=3.14159)), Declaration(class_='Declaration', name='tau', value=MathMul(class_='MathMul', expr1=MathNum(class_='MathNum', value=2), expr2=Access(class_='Access', name='pi'))), Declaration(class_='Declaration', name='omega', value=MathVar(c

In [8]:
from oqd_core.analysis.utils import cfg_to_dot
dot = cfg_to_dot(cfg)
print(dot.source)
dot.render("cfg", format="png", cleanup=True)


digraph {
	0 [label="0: start\nCFGStart"]
	0 -> 1
	1 [label="1: stmt\nr = ..."]
	1 -> 2
	2 [label="2: stmt\ns = ..."]
	2 -> 3
	3 [label="3: stmt\nq0 = ..."]
	3 -> 4
	4 [label="4: stmt\nq1 = ..."]
	4 -> 5
	5 [label="5: stmt\ntargets = ..."]
	5 -> 6
	6 [label="6: stmt\npi = ..."]
	6 -> 7
	7 [label="7: stmt\ntau = ..."]
	7 -> 8
	8 [label="8: stmt\nomega = ..."]
	8 -> 9
	9 [label="9: stmt\nphase = ..."]
	9 -> 10
	10 [label="10: stmt\nneg = ..."]
	10 -> 11
	11 [label="11: stmt\ncubed = ..."]
	11 -> 12
	12 [label="12: stmt\nsine = ..."]
	12 -> 13
	13 [label="13: stmt\ncosed = ..."]
	13 -> 14
	14 [label="14: stmt\ncplx = ..."]
	14 -> 15
	15 [label="15: stmt\nX = ..."]
	15 -> 16
	16 [label="16: stmt\nY = ..."]
	16 -> 17
	17 [label="17: stmt\nZ = ..."]
	17 -> 18
	18 [label="18: stmt\nI = ..."]
	18 -> 19
	19 [label="19: stmt\nC = ..."]
	19 -> 20
	20 [label="20: stmt\nA = ..."]
	20 -> 21
	21 [label="21: stmt\nJ = ..."]
	21 -> 22
	22 [label="22: stmt\nH_single = ..."]
	22 -> 23
	23 [label="23: stm

'cfg.png'

In [9]:
# from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
# out = compile_analog_circuit(circuit)
# print(printer(out))

In [10]:
# cfg = AnalogCFGBuilder().run(circuit)
# out = json.dumps({node_id: node.to_dict() for node_id, node in cfg.items()}, indent=2)
# print(out)


# print(symbol_table)
# symbol_table = str(symbol_table)
# tree = symbol_table.model_dump_json(indent=2, serialize_as_any=True)
# print(json.dumps(cfg, indent=2))